In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import tiktoken
import json
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
from groq import Groq
import os
from dotenv import load_dotenv
import random

load_dotenv()  # Load environment variables from a .env file if present
client = Groq(api_key=os.getenv("groq_api_key"))

d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
math =pd.read_parquet("../data/math-distillation/math_distillation_dataset.parquet")
code =pd.read_parquet("../data/code-distillation/code_distillation_dataset.parquet")
science =pd.read_parquet("../data/science/science_distillation_dataset.parquet")
common_sense =pd.read_parquet("../data/common-sense/commonsense_distillation_dataset.parquet")
creative_writing =pd.read_parquet("../data/creative-writing/creative_writing_distillation_dataset.parquet")
ideation =pd.read_parquet("../data/ideation/ideation_distillation_dataset.parquet")
financial = pd.read_parquet("../data/financial/financial_distillation_dataset.parquet")
reading = pd.read_parquet("../data/reading/reading_distillation_dataset.parquet")
summarization = pd.read_parquet("../data/summarization/summarization_distillation_dataset.parquet")
#Function that adds a column called split if it's not there and sets it to 'train'
def ensure_split_column(df):
    if 'split' not in df.columns:
        df['split'] = 'train'
    return df
math = ensure_split_column(math)
code = ensure_split_column(code)
science = ensure_split_column(science)
common_sense = ensure_split_column(common_sense)
creative_writing = ensure_split_column(creative_writing)
ideation = ensure_split_column(ideation)
financial = ensure_split_column(financial)
reading = ensure_split_column(reading)
summarization = ensure_split_column(summarization)


combined_df = pd.concat([math, code, science, common_sense, creative_writing, ideation, financial, reading, summarization], ignore_index=True)


In [3]:
##Function to clean noise in reasoning that was created either by the system prompt or indirectly created by model response

def clean_reasoning(text):
    # Handle None/NaN values
    if pd.isna(text) or text is None:
        return text

    # Clean up extra whitespace
    text = re.sub(r'\n\n+', '\n\n', text)
    text = re.sub(r'\s+\.', '.', text)  # Fix spacing before periods
    text = re.sub(r'\s+,', ',', text)  # Fix spacing before commas
    text = text.strip()
    
    # Only remove explicit "developer instructions" references
    #text = re.sub(r'developer instructions[:\s]*', '', text, flags=re.IGNORECASE)
    # Remove everything up to and including "developer instructions"
    text = re.sub(r'^.*?developer instructions[:\s]*', '', text, flags=re.IGNORECASE | re.MULTILINE)
    text= re.sub(r'^.*?"creative, specific, actionable ideas"[:\s]*', '', text, flags=re.IGNORECASE | re.MULTILINE)
    
    
    
    # Remove word count mentions - capture entire phrases
    text = re.sub(r'under \d+-\d+ words?\.?', '', text, flags=re.IGNORECASE)  # "under 200-300 words"
    text = re.sub(r'Keep (response |your response )?(under |within )?\d+-?\d* words?\.?', '', text, flags=re.IGNORECASE)  # "Keep 300-400 words", "Keep under 500 words"
    text = re.sub(r'\b\d+-\d+ words?\b', '', text)  # standalone "100-300 words"
    text = re.sub(r'~\d+ words?', '', text)          # "~180 words"
    text = re.sub(r'Word count target[^\n.]*\.?', '', text)
    text = re.sub(r'Should be under \d+\.?', '', text)
    text = re.sub(r'between \d+ and \d+ words?', '', text, flags=re.IGNORECASE)
    
    
    

    
    return text

In [31]:
##Adding a domain prefix to reasoning
##This creates a natural curriculum where the model learns distinct reasoning styles per domain.
##The domain prefix acts as a routing signal - essentially teaching the student model to self-identify the problem type and activate the appropriate reasoning mode.
def add_domain_prefix(reasoning, domain):
    """
    Add domain identification prefix to reasoning.
    
    Args:
        reasoning: The reasoning text
        domain: The domain (e.g., 'creative_ideation', 'summarization')
        
    Returns:
        str: Reasoning with domain prefix
    """
    if pd.isna(reasoning) or reasoning is None:
        return reasoning
    
    # Domain-specific prefixes
    domain_prefixes = {
        'code': 'This is a coding task.',
        'commonsense_reasoning': 'This is a commonsense reasoning task.',
        'creative_ideation': 'This is a creative ideation task.',
        'creative_writing': 'This is a creative writing task.',
        'financial_reasoning': 'This is a financial reasoning task.',
        'math': 'This is a mathematical reasoning task.',
        'numerical_reasoning': 'This is a numerical reasoning task.',
        'reading_comprehension': 'This is a reading comprehension task.',
        'science': 'This is a scientific reasoning task.',
        'summarization': 'This is a summarization task.'
    }
    
    # Get appropriate prefix
    prefix = domain_prefixes.get(domain, f'This is a {domain.replace("_", " ")} task.')
    
    # Add prefix to reasoning
    prefixed_reasoning = f"{prefix}\n\n{reasoning}"
    
    return prefixed_reasoning




##Filtering function to make sure we keep the content with quality reasoning
def is_low_quality_reasoning(reasoning, domain):
    # if domain != 'creative_ideation':
    #     return False
    
    if pd.isna(reasoning) or reasoning is None:
        return True
    
    # Remove ONLY meta-instruction phrases, not substantive content
    content = reasoning
    
    # Remove specific meta phrases (more precise patterns)
    meta_phrases = [
        r'We need to (provide|produce|create|predict|propose|brainstorm|respond|answer|generate)\b',
        r'Must be creative, specific, actionable\.?',
        r'Should be (within|under)\b[^.]*\.',
        r'Under \d+-\d+ words\.?',
        r'Keep (concise|within word limit)\.?',
        r'Provide (specific )?actionable ideas\.?',
        r'Also consider multiple perspectives\.?',
        r'Let\'s (craft|draft|produce)( about)?\.?\s*$',  # Only if it ends the text
        r'As a creative thinking expert,?',
    ]
    
    for phrase in meta_phrases:
        content = re.sub(phrase, '', content, flags=re.IGNORECASE)
    
    # Clean whitespace
    content = re.sub(r'\s+', ' ', content).strip()
    
    # Use character count (more reliable than word count)
    char_count = len(content)
    
    # Domain-specific thresholds
    thresholds = {
        'math': 80,  # Math reasoning is concise (calculations, formulas)
        'code': 80,  # Code reasoning can be brief (algorithm names, complexity)
        'numerical_reasoning': 80,
        'reading_comprehension': 100,
        'commonsense_reasoning': 100,
        'science': 100,
        'financial_reasoning': 150,
        'creative_ideation': 200,  # Creative needs more elaboration
        'creative_writing': 150,
        'summarization': 150,
    }
    
    threshold = thresholds.get(domain, 100)  # Default 100 chars
    
    if char_count < threshold:
        return True
    
    return False

In [32]:
##Clean the noise in reasoning
combined_df['cleaned_reasoning'] = combined_df['reasoning'].apply(clean_reasoning)

# Apply the function correctly
combined_df['low_reasoning_quality'] = combined_df.apply(
    lambda row: is_low_quality_reasoning(row['cleaned_reasoning'], row['domain']), 
    axis=1
)
# Add domain reasoning in
combined_df['cleaned_reasoning'] = combined_df.apply(
    lambda row: add_domain_prefix(row['cleaned_reasoning'], row['domain']),
    axis=1
)



In [33]:
combined_df.groupby(['domain','low_reasoning_quality']).size()

domain                 low_reasoning_quality
code                   False                     7933
                       True                        41
commonsense_reasoning  False                     7208
                       True                        82
creative_ideation      False                     1819
                       True                       181
creative_writing       False                     3844
                       True                         2
financial_reasoning    False                     1674
                       True                       100
math                   False                    13104
                       True                      1718
numerical_reasoning    False                     2000
reading_comprehension  False                      969
                       True                        31
science                False                     4524
                       True                        42
summarization          False         

In [ ]:
# temp=combined_df[combined_df['low_reasoning_quality']==True][['input','domain','reasoning','cleaned_reasoning','response']].sample(100)
# temp.to_csv('sample_to_clean.csv')
# temp

,input,domain,reasoning,cleaned_reasoning,response
25262,Question: What are two types of lobe finned fi...,science,"We need to answer the question: ""What are two ...",This is a scientific reasoning task.\n\nWe nee...,Lobe‑finned fishes (class **Sarcopterygii**) a...
1646,198 passengers fit into 9 buses. How many pass...,math,We need to solve proportion: 198 passengers / ...,This is a mathematical reasoning task.\n\nWe n...,The number of passengers per bus is constant.\...
8012,My wife wants to evenly split the check but wa...,math,None,None,None
20129,Below is an instruction that describes a task....,code,We need to respond with a solution: explain ap...,This is a coding task.\n\nWe need to respond w...,**Solution Explanation**\n\nThe task is to tur...
45215,Context: The Olympic Games - Facts & Summary -...,reading_comprehension,"The user asks: ""Where were the 2004 Summer Oly...",This is a reading comprehension task.\n\nThe u...,The 2004 Summer Olympic Games were held in **A...
...,...,...,...,...,...
971,Annabelle collected a weekly allowance of $30....,math,We need to compute: allowance = $30. She spent...,This is a mathematical reasoning task.\n\nWe n...,**Step‑by‑step calculation**\n\n1. **Total wee...
21554,Below is an instruction that describes a task....,code,The user wants a response that appropriately c...,This is a coding task.\n\nexplain solution app...,**Solution Approach**\n\n1. Store the supplied...
7940,Bob is in charge of doing laundry for a large ...,math,None,None,None
43237,the following table illustrates the effect tha...,financial_reasoning,We need to extract from table: fair value of f...,This is a financial reasoning task.\n\nWe need...,The fair value of the forward‑exchange‑contrac...


In [ ]:
# temp2=combined_df[combined_df['low_reasoning_quality']==True][['input','domain','reasoning','cleaned_reasoning','response']]
# temp2[temp2['domain']=='science'].sample(20)

,input,domain,reasoning,cleaned_reasoning,response
25244,Question: What is known as the universal solve...,science,We need answer: water is called universal solv...,This is a scientific reasoning task.\n\nWe nee...,**Answer: A. water**\n\n**Explanation**\n\nWat...
23347,Question: Water evaporates and falls back to E...,science,We need to answer: primary energy source that ...,This is a scientific reasoning task.\n\nWe nee...,The water (hydrologic) cycle is powered mainly...
26557,Question: What is the bending of waves around ...,science,We need to answer: bending of waves around a c...,This is a scientific reasoning task.\n\nWe nee...,The phenomenon where a wave spreads out as it ...
24257,Question: What is our main source of aluminum ...,science,We need to answer: main source of aluminum ore...,This is a scientific reasoning task.\n\nWe nee...,**Answer: A. bauxite**\n\n**Explanation:** \n...
25359,Question: What is the term for animals that ea...,science,We need to answer: term for animals that eat p...,This is a scientific reasoning task.\n\nWe nee...,Animals that obtain their energy by eating pro...
25218,Question: How many chromosomes are in a diploi...,science,We need to answer: diploid human cell has 46 c...,This is a scientific reasoning task.\n\nWe nee...,"In humans, the normal somatic (non‑reproductiv..."
24933,Question: What is the term for gases that abso...,science,We need to answer: term for gases that absorb ...,This is a scientific reasoning task.\n\nWe nee...,The gases that trap infrared radiation and thu...
25565,Question: What do we call the lowest layer of ...,science,We need to answer: lowest layer of atmosphere ...,This is a scientific reasoning task.\n\nWe nee...,The atmosphere is divided into several layers ...
25656,Question: What is the largest organ of the hum...,science,We need answer with explanation. The largest o...,This is a scientific reasoning task.\n\nWe nee...,The largest organ in the human body is **the s...
24703,Question: The science of analyzing tree rings ...,science,We need answer: dendrochronology. Provide expl...,This is a scientific reasoning task.\n\nWe nee...,The study that examines the patterns of growth...


In [36]:
##Filter to train data samples and those with reasoning and response populated
combined_df1 = combined_df[(combined_df['split'] == 'train') & 
                          (combined_df['cleaned_reasoning'].notnull()) & 
                          (combined_df['response'].notnull())]

print(f"Filtered dataset size: {len(combined_df1)}")

#Filter to data with total_tokens < 1800
combined_df2 = combined_df1[combined_df1['total_tokens'] < 1800]

##Filter ideation data with low reasoning content
combined_df2 = combined_df2[combined_df2['low_reasoning_quality'] ==False]
print(f"Filtered dataset size after token limit: {len(combined_df2)}")



Filtered dataset size: 45233
Filtered dataset size after token limit: 39289


In [37]:
print(f"Total combined dataset size: {len(combined_df2)}")
print("Dataset sizes by domain:")
for domain in combined_df2['domain'].unique():
    domain_size = len(combined_df2[combined_df2['domain'] == domain])
    print(f"  {domain}: {domain_size}")
    
print("Dataset sizes by source:")
for source in combined_df2['source'].unique():
    source_size = len(combined_df2[combined_df2['source'] == source])
    print(f"  {source}: {source_size}")

Total combined dataset size: 39289
Dataset sizes by domain:
  math: 10385
  code: 6348
  science: 4322
  commonsense_reasoning: 6372
  creative_writing: 3825
  creative_ideation: 1797
  numerical_reasoning: 1997
  financial_reasoning: 1665
  reading_comprehension: 964
  summarization: 1614
Dataset sizes by source:
  gsm8k: 7423
  OpenR1-Math-220k: 2317
  Math/AIME 2025: 5
  ChilleD/SVAMP: 640
  google-research-datasets/mbpp: 324
  nvidia/OpenCodeReasoning: 100
  codeparrot/apps: 107
  sahil2801/CodeAlpaca-20k: 2945
  iamtarun/python_code_instructions_18k_alpaca: 2872
  ai2_arc/ARC-Challenge: 1115
  sciq: 2962
  Idavidrein/gpqa: 245
  tau/commonsense_qa: 1999
  tasksource/strategy-qa: 2281
  hotpot_qa: 2092
  euclaise/writingprompts: 1830
  igormorgado/ROCStories2018: 1995
  synthetic_ideation_prompts: 1797
  ibm/tatqa: 1997
  FinGPT/fingpt-convfinqa: 1665
  ehovy/race: 493
  FabianWillner/triviaQARC: 471
  abisee/cnn_dailymail: 1614


##Train - Validation - Test Split

In [38]:
from sklearn.model_selection import train_test_split

# Samples filtered out because split != 'train'
non_train_split = combined_df[(combined_df['split'] != 'train') & 
                               (combined_df['reasoning'].notnull()) & 
                               (combined_df['response'].notnull())]
print(f"Samples from non-train splits: {len(non_train_split)}")

# Samples that passed initial filters but exceeded token limit
dropped_samples = combined_df1[combined_df1['total_tokens'] >= 1900]
print(f"Samples dropped due to token limit: {len(dropped_samples)}")

# Your filtered training candidates
combined_df2 = combined_df1[combined_df1['total_tokens'] < 1900]
print(f"Samples for train/valid/test split: {len(combined_df2)}")

# First split: 90% train, 10% temp (which will become 5% valid, 5% test)
train_df, temp_df = train_test_split(
    combined_df2, 
    train_size=0.95, 
    stratify=combined_df2['domain'],
    random_state=42
)

# Second split: split the 10% temp into 50-50 (5% valid, 5% test of original)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.7,
    stratify=temp_df['domain'],
    random_state=42
)

# Add dropped samples and non-train splits to test set
test_df = pd.concat([test_df, dropped_samples, non_train_split], ignore_index=True)

print(f"\nFinal split:")
print(f"Train: {len(train_df)} ({len(train_df)/len(combined_df2)*100:.1f}%)")
print(f"Valid: {len(valid_df)} ({len(valid_df)/len(combined_df2)*100:.1f}%)")
print(f"Test: {len(test_df)} samples (base: {len(temp_df)//2}, +{len(dropped_samples)} long-context, +{len(non_train_split)} non-train splits)")

Samples from non-train splits: 600
Samples dropped due to token limit: 4645
Samples for train/valid/test split: 40588

Final split:
Train: 38558 (95.0%)
Valid: 609 (1.5%)
Test: 6666 samples (base: 1015, +4645 long-context, +600 non-train splits)


In [40]:
print(f"Total combined dataset size: {len(train_df)}")
print("Dataset sizes by domain:")
for domain in train_df['domain'].unique():
    domain_size = len(train_df[train_df['domain'] == domain])
    print(f"  {domain}: {domain_size}")

Total combined dataset size: 38558
Dataset sizes by domain:
  financial_reasoning: 1682
  math: 10118
  commonsense_reasoning: 6356
  science: 4161
  code: 6151
  creative_writing: 3637
  summarization: 1715
  reading_comprehension: 947
  creative_ideation: 1893
  numerical_reasoning: 1898


In [39]:
train_df.to_parquet('../data//final_data//custom_df_train_sft.parquet')
valid_df.to_parquet('../data//final_data//custom_df_valid_sft.parquet')
test_df.to_parquet('../data//final_data//custom_df_test_sft.parquet')

In [15]:
train_df.sample(5)

,input,source_answer,split,source,domain,ground_truth,problem_type,question_type,uid,reasoning,...,model_answer_available,validated_correct,difficulty,input_token_count,extracted_answer,is_correct,story_token_count,category,cleaned_reasoning,low_reasoning_quality
3132,Bald Mountain received 1.5 meters of snow last...,Bald Mountain = 1.5 * 100 = <<1.5*100=150>>150...,train,gsm8k,math,326,,,math3133,We need to parse problem: Bald Mountain got 1....,...,True,False,NaN,NaN,NaN,NaN,NaN,NaN,This is a mathematical reasoning task.\n\nWe n...,False
24547,Question: What are the rolling motions during ...,A,train,sciq,science,A,multiple_choice,multiple_choice,science1752,"We need to answer the question: ""What are the ...",...,NaN,NaN,NaN,35.0,NaN,NaN,NaN,NaN,This is a scientific reasoning task.\n\nWe nee...,False
34281,Context:\nRobert Smith (musician): Robert Jame...,Robert Smith,train,hotpot_qa,commonsense_reasoning,Robert Smith,open_ended,medium,commonsense6920,"We need to answer: ""What guitarist worked with...",...,NaN,NaN,NaN,NaN,Robert Smith,False,NaN,NaN,This is a commonsense reasoning task.\n\nWe ne...,False
42770,measurement point december 31 the priceline gr...,33.73,train,FinGPT/fingpt-convfinqa,financial_reasoning,33.73,financial_qa,Conversational financial QA,dquant2273,We need to compute difference: Priceline Group...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,This is a financial reasoning task.\n\nWe need...,False
17842,Write a PHP code for sending an email to the c...,$to = 'xyz@example.com';\n$subject = 'Subject ...,train,sahil2801/CodeAlpaca-20k,code,,code generation,code generation by instruction,code23174,We need to write PHP code to send an email to ...,...,NaN,NaN,UNKNOWN_DIFFICULTY,NaN,NaN,NaN,NaN,NaN,This is a coding task.\n\nWe need to write PHP...,False


In [3]:
os.getcwd()

'd:\\Work\\Github\\google-tunix-kaggle\\data-generation'

In [ ]:
valid_df = pd.read_parquet("../data//final_data//custom_df_valid_sft.parquet")
# Define your target counts per domain
domain_counts = {
    'code': 10,
    'math': 10,
    'commonsense_reasoning': 20,
    'creative_ideation': 20,
    'creative_writing': 20,
    'financial_reasoning': 10,
    'numerical_reasoning': 10,
    'reading_comprehension': 20,
    'science': 20,
    'summarization': 20
}

# Sample each domain separately and concatenate
sampled_dfs = []
for domain, count in domain_counts.items():
    domain_samples = valid_df[valid_df['domain'] == domain].sample(
        n=min(count, len(valid_df[valid_df['domain'] == domain])),
        random_state=42  # for reproducibility
    )
    sampled_dfs.append(domain_samples)

valid_df_eval_sample = pd.concat(sampled_dfs, ignore_index=True)
valid_df_eval_sample.groupby('domain').size()

domain
code                     10
commonsense_reasoning    20
creative_ideation        20
creative_writing         20
financial_reasoning      10
math                     10
numerical_reasoning      10
reading_comprehension    20
science                  20
summarization            20
dtype: int64

In [11]:
valid_df_eval_sample=valid_df_eval_sample[['uid','input','ground_truth','domain','problem_type','reasoning','response']]
valid_df_eval_sample.to_parquet("../data//final_data//model_evaluation_sample.parquet")

In [13]:
print(valid_df_eval_sample.groupby('domain').size())

domain
code                     10
commonsense_reasoning    20
creative_ideation        20
creative_writing         20
financial_reasoning      10
math                     10
numerical_reasoning      10
reading_comprehension    20
science                  20
summarization            20
dtype: int64


In [16]:
valid_df_eval_sample.sample(10)

,uid,input,ground_truth,domain,problem_type,reasoning,response
95,dquant1373,"Table :\n"""","""","""",""Total Number of"",""Approxima...",(29.1-58.0)/58.0,numerical_reasoning,calculation,We need to extract numbers: March total number...,**Relevant numbers from the table**\n\n| Month...
30,commonsense1033,"The forgotten leftovers had gotten quite old, ...",B,commonsense_reasoning,multiple_choice,"We need to answer the question: ""The forgotten...",**B. refrigerator**
0,code18321,How can we check whether a given number is pri...,,code,code generation,"We need to answer: explain solution approach, ...",**Solution Approach**\n\n1. **Handle edge case...
14,math4544,Sarah interviewed 450 students at her school a...,113,math,,We need to find number who liked strawberries....,"Since every student chose exactly one fruit, t..."
133,science1890,"Question: Starch and glycogen, as well as simp...",A,science,multiple_choice,"The user asks: ""Starch and glycogen, as well a...","Starch, glycogen, glucose, and fructose are al..."
4,code25223,What is the output of the following Java code?...,,code,code generation,We need to answer: What is output? The code de...,**Explanation**\n\n- `x % y` is the **modulo (...
18,math6316,Jack is mad at his neighbors for blasting Tayl...,1450,math,,We need to compute damages: 3 tires at $250 ea...,Jack’s damages consist of:\n\n- **Tires:** 3 ×...
116,reading329,"Context: Wanted, Someone for a Kiss\nWe're loo...",D,reading_comprehension,reading_multiple_choice,We need to answer based on the ad for Father C...,**Answer: D. can work for about a month.**
135,science2029,Question: What measurement is used to describe...,A,science,multiple_choice,"We need to answer the question: ""What measurem...",The amount of water vapor present in the atmos...
57,ideation757,Combine wine corks with sensors to create some...,None,creative_ideation,brainstorming,User asks to combine wine corks with sensors t...,"**Product Concept: “CorkSense™ – Smart, Self‑P..."
